# Historical photon-transport experiments
Original code cells retained for provenance; outputs and execution metadata removed.
**Some cells require substantial runtime and memory and contain known model defects.**
Use `notebooks/transport_demo.ipynb` for the maintained, bounded example.
See `docs/REVIEW.md` before running historical experiments.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# --- Parâmetros do voxel e volume ---
voxel_size = 0.1  # cm = 1 mm
volume_dims = (100, 100, 100)  # 10x10x10 cm³ => (100x100x100 voxels)
energy_grid = np.zeros(volume_dims)  # matriz 3D [x, y, z]

# --- Tabela simplificada de coeficientes para Pb ---
energy_levels = np.array([5E-02,1E-01,5E-01,1E00])
mu_total = np.array([7.787E00, 5.336E+00, 1.499E-01, 6.803E-02])     # cm²/g
mu_photo = np.array([7.292E+00, 5.237E+00, 8.257E-02, 1.810E-02])
mu_compton = np.array([9.478E-02 , 9.894E-02, 6.734E-02, 4.993E-02])
mu_pair = np.array([0, 0, 0, 0])

rho = 11.34  # g/cm³ Pb

# --- Tabela simplificada de coeficientes para H2O ---
#energy_levels = np.array([5E-02,1E-01,5E-01,1E00])
#mu_total = np.array([2.076E-01 , 1.654E-01, 9.665E-02 , 7.066E-02 ])     # cm²/g
#mu_photo = np.array([2.725E-02, 2.763E-03, 1.883E-05, 3.681E-06])
#mu_compton = np.array([1.803E-01 , 1.626E-01, 9.663E-02, 7.066E-02])
#mu_pair = np.array([0, 0, 0, 0])

#rho = 1  # g/cm³ Pb

# --- Interpoladores ---
get_mu_total = interp1d(energy_levels, mu_total, fill_value="extrapolate")
get_mu_photo = interp1d(energy_levels, mu_photo, fill_value="extrapolate")
get_mu_compton = interp1d(energy_levels, mu_compton, fill_value="extrapolate")
get_mu_pair = interp1d(energy_levels, mu_pair, fill_value="extrapolate")

def sample_compton_angle(E):
    mec2 = 0.511
    while True:
        cos_theta = np.random.uniform(-1, 1)
        theta = np.arccos(cos_theta)
        E_prime = E / (1 + (E / mec2) * (1 - cos_theta))
        P = E_prime**2 / E**2 * (E / E_prime + E_prime / E - np.sin(theta)**2)
        if np.random.uniform(0, 2) < P:
            return theta

def position_to_voxel(pos):
    idx = np.floor(pos / voxel_size).astype(int)
    if np.any(idx < 0) or np.any(idx >= np.array(volume_dims)):
        return None  # fora dos limites
    return tuple(idx)

for N in range(10000):
    
    #print(N)
    # --- Inicialização do fóton ---    
    E0 = 1.0  # MeV
    photon_energy = E0
    photon_pos = np.array([5, 5, 0.0])  # cm
    photon_dir = np.array([0.0, 0.0, 1.0])  # para z+
    
    while 0 <= photon_pos[2] < volume_dims[2] * voxel_size and photon_energy > 0.01:
        mu_tot = get_mu_total(photon_energy) * rho
        mu_ph = get_mu_photo(photon_energy)
        mu_cpt = get_mu_compton(photon_energy)
        mu_ppr = get_mu_pair(photon_energy)
        probs = np.array([mu_ph, mu_cpt, mu_ppr]) / (mu_ph + mu_cpt + mu_ppr)
    
        step = np.random.exponential(1 / mu_tot)
        photon_pos += photon_dir * step
    
        interaction = np.random.choice(['photo', 'compton', 'pair'], p=probs)
        voxel = position_to_voxel(photon_pos)
        if voxel is None:
            break
    
        if interaction == 'photo':
            energy_grid[voxel] += photon_energy
            break
    
        elif interaction == 'compton':
            theta = sample_compton_angle(photon_energy)
            phi = np.random.uniform(0, 2*np.pi)
            mec2 = 0.511
            E = photon_energy
            E_prime = E / (1 + (E / mec2) * (1 - np.cos(theta)))
            dE = E - E_prime
            energy_grid[voxel] += dE
            photon_energy = E_prime
    
            # Atualizar direção em 3D
            u = photon_dir
            if np.allclose(u, [0, 0, 1]):
                v = np.array([1, 0, 0])
            else:
                v = np.cross(u, [0, 0, 1])
                v /= np.linalg.norm(v)
            w = np.cross(u, v)
            new_dir = (np.sin(theta) * np.cos(phi) * v +
                       np.sin(theta) * np.sin(phi) * w +
                       np.cos(theta) * u)
            photon_dir = new_dir / np.linalg.norm(new_dir)
    
        elif interaction == 'pair':
            if photon_energy > 1.022:
                deposited = photon_energy - 1.022
                energy_grid[voxel] += deposited
                break

# --- Visualização: Perfil de dose ao longo de profundidade (z) ---
depth_profile = energy_grid.sum(axis=(0, 1))  # soma em x e y
plt.figure(figsize=(6, 4))

z_axis = np.arange(volume_dims[2]) * voxel_size
plt.plot(z_axis, depth_profile)
plt.xlabel('Profundidade z [cm]')
plt.ylabel('Energia depositada [MeV]')
plt.title('Perfil de energia depositada (1 fóton)')
plt.grid(True)
plt.show()


In [ ]:
mid_axis1 = energy_grid.shape[0] // 2
mid_axis2 = energy_grid.shape[1] // 2
slice_1d = energy_grid[mid_axis1, mid_axis2, :]

In [ ]:
plt.plot(z_axis, slice_1d/np.max(slice_1d))
plt.plot(z_axis, np.exp(-mu_total[-1]*z_axis*rho))
plt.xlabel('Profundidade z [cm]')
plt.ylabel('Energia depositada [MeV]')
plt.title('Perfil de energia depositada (1 fóton)')
plt.grid(True)
plt.show()

In [ ]:
mid_axis1 = energy_grid.shape[0] // 2
mid_axis2 = energy_grid.shape[2] // 5
slice_1d = energy_grid[mid_axis1, :, mid_axis2]

In [ ]:
y_axis = np.arange(volume_dims[1]) * voxel_size
plt.plot(y_axis, slice_1d)
plt.xlim(4.5,5.5)
plt.xlabel('Profundidade y [cm]')
plt.ylabel('Energia depositada [MeV]')
plt.title('Perfil de energia depositada (1 fóton)')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mid0 = energy_grid.shape[0] // 2
mid1 = energy_grid.shape[1] // 2
mid2 = energy_grid.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid[mid0, :, :]  # plano YZ (fixando X)

# Plotar os planos com colormap
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# XY
axes[0].imshow(slice_xy, cmap='jet', origin='lower', aspect='auto')
axes[0].set_title("Plano XY (Z fixo)")
axes[0].axis('on')
plt.colorbar(axes[0].imshow(slice_xy, cmap='jet'), ax=axes[0])

# XZ
axes[1].imshow(slice_xz, cmap='jet', origin='lower', aspect='auto')
axes[1].set_title("Plano XZ (Y fixo)")
axes[1].axis('on')
plt.colorbar(axes[1].imshow(slice_xz, cmap='jet'), ax=axes[1])

# YZ
axes[2].imshow(slice_yz, cmap='jet', origin='lower', aspect='auto')
axes[2].set_title("Plano YZ (X fixo)")
axes[2].axis('on')
plt.colorbar(axes[2].imshow(slice_yz, cmap='jet'), ax=axes[2])

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

energy_grid_log = energy_grid + 1e-6
mid0 = energy_grid.shape[0] // 2
mid1 = energy_grid.shape[1] // 2
mid2 = energy_grid.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid[mid0, :, :]  # plano YZ (fixando X)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Usar LogNorm para escala logarítmica
norm = LogNorm(vmin=np.min(energy_grid_log), vmax=np.max(energy_grid_log))

# XY
im0 = axes[0].imshow(slice_xy, cmap='jet', norm=norm, origin='lower', aspect='auto')
axes[0].set_title("Plano XY (Z fixo)")
plt.colorbar(im0, ax=axes[0])

# XZ
im1 = axes[1].imshow(slice_xz, cmap='jet', norm=norm, origin='lower', aspect='auto')
axes[1].set_title("Plano XZ (Y fixo)")
plt.colorbar(im1, ax=axes[1])

# YZ
im2 = axes[2].imshow(slice_yz, cmap='jet', norm=norm, origin='lower', aspect='auto')
axes[2].set_title("Plano YZ (X fixo)")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Example grid
grid = np.log(energy_grid_log)

fig = go.Figure(data=go.Volume(
    x=np.repeat(np.arange(grid.shape[0]), grid.shape[1] * grid.shape[2]),
    y=np.tile(np.repeat(np.arange(grid.shape[1]), grid.shape[2]), grid.shape[0]),
    z=np.tile(np.arange(grid.shape[2]), grid.shape[0] * grid.shape[1]),
    value=grid.flatten(),
    opacity=0.1,  # overall opacity of the volume
    surface_count=10,  # number of isosurfaces
    colorscale='Jet',
))

fig.update_layout(scene=dict(
    xaxis_title='X',
    yaxis_title='Y',
    zaxis_title='Z'
))
fig.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# --- Tabela simplificada de coeficientes para Pb ---
energy_levels = np.array([5E-02,1E-01,5E-01,1E00])
mu_total = np.array([7.787E00, 5.336E+00, 1.499E-01, 6.803E-02])     # cm²/g
mu_photo = np.array([7.292E+00, 5.237E+00, 8.257E-02, 1.810E-02])
mu_compton = np.array([9.478E-02 , 9.894E-02, 6.734E-02, 4.993E-02])
mu_pair = np.array([0, 0, 0, 0])

rho = 11.34  # g/cm³ Pb

# --- Tabela simplificada de coeficientes para H2O ---
#energy_levels = np.array([5E-02,1E-01,5E-01,1E00])
#mu_total = np.array([2.076E-01 , 1.654E-01, 9.665E-02 , 7.066E-02 ])     # cm²/g
#mu_photo = np.array([2.725E-02, 2.763E-03, 1.883E-05, 3.681E-06])
#mu_compton = np.array([1.803E-01 , 1.626E-01, 9.663E-02, 7.066E-02])
#mu_pair = np.array([0, 0, 0, 0])

#rho = 1  # g/cm³ Pb

# --- Interpoladores ---
get_mu_total = interp1d(energy_levels, mu_total, fill_value="extrapolate")
get_mu_photo = interp1d(energy_levels, mu_photo, fill_value="extrapolate")
get_mu_compton = interp1d(energy_levels, mu_compton, fill_value="extrapolate")
get_mu_pair = interp1d(energy_levels, mu_pair, fill_value="extrapolate")

def sample_compton_angle(E):
    mec2 = 0.511
    while True:
        cos_theta = np.random.uniform(-1, 1)
        theta = np.arccos(cos_theta)
        E_prime = E / (1 + (E / mec2) * (1 - cos_theta))
        P = E_prime**2 / E**2 * (E / E_prime + E_prime / E - np.sin(theta)**2)
        if np.random.uniform(0, 2) < P:
            return theta

def position_to_voxel(pos):
    idx = np.floor(pos / voxel_size).astype(int)
    if np.any(idx < 0) or np.any(idx >= np.array(volume_dims)):
        return None  # fora dos limites
    return tuple(idx)

num_histories = 200

# --- Parâmetros do voxel e volume ---
voxel_size = 0.1  # cm = 1 mm
volume_dims = (100, 100, 100)  # 10x10x10 cm³ => (100x100x100 voxels)
energy_grid = np.zeros(volume_dims)  # matriz 3D [x, y, z]
energy_grid_record = np.zeros((num_histories, *volume_dims))

for i in range (num_histories):
    energy_grid = np.zeros(volume_dims)

    perc = i/num_histories*100
    print(f"{np.round(perc,1)}% of the Simulation Done")
    for N in range(10000):
        
        #print(N)
        # --- Inicialização do fóton ---
        
        E0 = 1.0  # MeV
        photon_energy = E0
        photon_init = [(np.array([5.0, 5.0, -0.1]),np.array([0.0, 0.0, 1.0])),
                       (np.array([5.0, 5.0, 10.1]),np.array([0.0, 0.0, -1.0])),
                       (np.array([5.0, 10.1, 5.0]),np.array([0.0, -1.0, 0.0])),
                       (np.array([5.0, -0.1, 5.0]),np.array([0.0, 1.0, 0.0])),
                       (np.array([5.0, 10.1, 10.1]),np.array([0.0, -1.0, -1.0])),
                       (np.array([5.0, -0.1, 10.1]),np.array([0.0, 1.0, -1.0])),
                       (np.array([5.0, 10.1, -0.1]),np.array([0.0, -1.0, 1.0])),
                       (np.array([5.0, -0.1, -0.1]),np.array([0.0, 1.0, 1.0]))
                      ]
        
        for (photon_pos, photon_dir) in photon_init:
            
            while (-0.1 <= photon_pos[0] <= (volume_dims[0]+1) * voxel_size) and (-0.1 <= photon_pos[1] <= (volume_dims[1]+1) * voxel_size) and (-0.1 <= photon_pos[2] <= (volume_dims[2]+1) * voxel_size) and photon_energy > 0.01:
                mu_tot = get_mu_total(photon_energy) * rho
                mu_ph = get_mu_photo(photon_energy)
                mu_cpt = get_mu_compton(photon_energy)
                mu_ppr = get_mu_pair(photon_energy)
                probs = np.array([mu_ph, mu_cpt, mu_ppr]) / (mu_ph + mu_cpt + mu_ppr)
            
                step = np.random.exponential(1 / mu_tot)
                photon_pos += photon_dir * step
            
                interaction = np.random.choice(['photo', 'compton', 'pair'], p=probs)
                voxel = position_to_voxel(photon_pos)
                if voxel is None:
                    break
            
                if interaction == 'photo':
                    energy_grid[voxel] += photon_energy
                    break
            
                elif interaction == 'compton':
                    theta = sample_compton_angle(photon_energy)
                    phi = np.random.uniform(0, 2*np.pi)
                    mec2 = 0.511
                    E = photon_energy
                    E_prime = E / (1 + (E / mec2) * (1 - np.cos(theta)))
                    dE = E - E_prime
                    energy_grid[voxel] += dE
                    photon_energy = E_prime
            
                    # Atualizar direção em 3D
                    u = photon_dir
                    if np.allclose(u, [0, 0, 1]) or np.allclose(u, [0, 0, -1]):
                        v = np.array([1, 0, 0])
                    else:
                        v = np.cross(u, [0, 0, 1])
                        v /= np.linalg.norm(v)
                    w = np.cross(u, v)
                    new_dir = (np.sin(theta) * np.cos(phi) * v +
                               np.sin(theta) * np.sin(phi) * w +
                               np.cos(theta) * u)
                    photon_dir = new_dir / np.linalg.norm(new_dir)
            
                elif interaction == 'pair':
                    if photon_energy > 1.022:
                        deposited = photon_energy - 1.022
                        energy_grid[voxel] += deposited
                        break
                        
        energy_grid_record[i] = energy_grid.copy()

print('End of Simulation')


In [ ]:
sum_energy_grid = energy_grid_record.sum(axis=(0))

energy_grid = sum_energy_grid

mid_axis1 = energy_grid.shape[0] // 2
mid_axis2 = energy_grid.shape[1] // 2
slice_1d = energy_grid[mid_axis1, mid_axis2, :]

y_axis = np.arange(volume_dims[1]) * voxel_size
plt.plot(y_axis, slice_1d)
plt.xlabel('Z-axis Length (cm)')
plt.ylabel('Energy (MeV)')
plt.title('Z-axis Profile')
# plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

energy_grid = sum_energy_grid

mid0 = energy_grid.shape[0] // 2
mid1 = energy_grid.shape[1] // 2
mid2 = energy_grid.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid[mid0, :, :]  # plano YZ (fixando X)

# Plotar os planos com colormap
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Sum of Dose Distributions (linear scale)')

# XY
axes[0].imshow(slice_xy, cmap='jet', origin='lower', aspect='auto',label='Energy (MeV)')
axes[0].set_title("Central XY Plan (mid Z)")
axes[0].axis('on')
plt.colorbar(axes[0].imshow(slice_xy, cmap='jet'), ax=axes[0])
plt.xlabel("X-axis Length (cm)")
plt.ylabel("Y-axis Length (cm)")

# XZ
axes[1].imshow(slice_xz, cmap='jet', origin='lower', aspect='auto',label='Energy (MeV)')
axes[1].set_title("Central XZ Plan (mid Y)")
axes[1].axis('on')
plt.colorbar(axes[1].imshow(slice_xz, cmap='jet'), ax=axes[1])
plt.xlabel("Y-axis Length (cm)")
plt.ylabel("Z-axis Length (cm)")

# YZ
axes[2].imshow(slice_yz, cmap='jet', origin='lower', aspect='auto',label='Energy (MeV)')
axes[2].set_title("Central YZ Plan (mid X)")
axes[2].axis('on')
plt.colorbar(axes[2].imshow(slice_yz, cmap='jet'), ax=axes[2])
plt.xlabel("Y-axis Length (cm)")
plt.ylabel("Z-axis Length (cm)")
           
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

energy_grid_log = sum_energy_grid + 1e-3

mid0 = energy_grid_log.shape[0] // 2
mid1 = energy_grid_log.shape[1] // 2
mid2 = energy_grid_log.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid_log[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid_log[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid_log[mid0, :, :]  # plano YZ (fixando X)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Sum of Dose Distributions (log scale)')

# Usar LogNorm para escala logarítmica
norm = LogNorm(vmin=np.min(energy_grid_log), vmax=np.max(energy_grid_log))

# XY
im0 = axes[0].imshow(slice_xy, cmap='jet', norm=norm, origin='lower', aspect='auto',label='Energy (MeV)')
axes[0].set_title("Central XY Plan (mid Z)")
plt.colorbar(im0, ax=axes[0])
plt.xlabel("X-axis Length (cm)")
plt.ylabel("Y-axis Length (cm)")

# XZ
im1 = axes[1].imshow(slice_xz, cmap='jet', norm=norm, origin='lower', aspect='auto',label='Energy (MeV)')
axes[1].set_title("Central XZ Plan (mid Y)")
plt.colorbar(im1, ax=axes[1])
plt.xlabel("Y-axis Length (cm)")
plt.ylabel("Z-axis Length (cm)")

# YZ
im2 = axes[2].imshow(slice_yz, cmap='jet', norm=norm, origin='lower', aspect='auto',label='Energy (MeV)')
axes[2].set_title("Central YZ (mid X)")
plt.colorbar(im2, ax=axes[2])
plt.xlabel("Y-axis Length (cm)")
plt.ylabel("Z-axis Length (cm)")

plt.tight_layout()
plt.show()

In [ ]:
mean_energy_grid_record = np.mean(energy_grid_record, axis=0)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

energy_grid_log = mean_energy_grid_record + 1e-3
mid0 = energy_grid_log.shape[0] // 2
mid1 = energy_grid_log.shape[1] // 2
mid2 = energy_grid_log.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid_log[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid_log[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid_log[mid0, :, :]  # plano YZ (fixando X)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Average Dose Distributions (log scale)')

# Usar LogNorm para escala logarítmica
norm = LogNorm(vmin=np.min(energy_grid_log), vmax=np.max(energy_grid_log))

# XY
im0 = axes[0].imshow(slice_xy, cmap='jet', norm=norm, origin='lower', aspect='auto',label='Energy (MeV)')
axes[0].set_title("Central XY Plan (mid Z)")
plt.colorbar(im0, ax=axes[0])
plt.xlabel("X-axis Length (cm)")
plt.ylabel("Y-axis Length (cm)")

# XZ
im1 = axes[1].imshow(slice_xz, cmap='jet', norm=norm, origin='lower', aspect='auto',label='Energy (MeV)')
axes[1].set_title("Central XZ Plan (mid Y)")
plt.colorbar(im1, ax=axes[1])
plt.xlabel("Y-axis Length (cm)")
plt.ylabel("Z-axis Length (cm)")

# YZ
im2 = axes[2].imshow(slice_yz, cmap='jet', norm=norm, origin='lower', aspect='auto',label='Energy (MeV)')
axes[2].set_title("Central YZ (mid X)")
plt.colorbar(im2, ax=axes[2])
plt.xlabel("Y-axis Length (cm)")
plt.ylabel("Z-axis Length (cm)")

plt.tight_layout()
plt.show()

In [ ]:
var_energy_grid_record = np.var(energy_grid_record, axis=0, ddof=1)
std_err_energy_grid_record = np.sqrt(var_energy_grid_record / energy_grid_record.shape[0])

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

energy_grid = std_err_energy_grid_record

mid0 = energy_grid.shape[0] // 2
mid1 = energy_grid.shape[1] // 2
mid2 = energy_grid.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid[mid0, :, :]  # plano YZ (fixando X)

# Plotar os planos com colormap
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Standard error of Dose Distributions (linear scale)')

# XY
axes[0].imshow(slice_xy, cmap='jet', origin='lower', aspect='auto',label='Energy (MeV)')
axes[0].set_title("Central XY Plan (mid Z)")
axes[0].axis('on')
plt.colorbar(axes[0].imshow(slice_xy, cmap='jet'), ax=axes[0])
plt.xlabel("X-axis Length (cm)")
plt.ylabel("Y-axis Length (cm)")

# XZ
axes[1].imshow(slice_xz, cmap='jet', origin='lower', aspect='auto',label='Energy (MeV)')
axes[1].set_title("Central XZ Plan (mid Y)")
axes[1].axis('on')
plt.colorbar(axes[1].imshow(slice_xz, cmap='jet'), ax=axes[1])
plt.xlabel("Y-axis Length (cm)")
plt.ylabel("Z-axis Length (cm)")

# YZ
axes[2].imshow(slice_yz, cmap='jet', origin='lower', aspect='auto',label='Energy (MeV)')
axes[2].set_title("Central YZ Plan (mid X)")
axes[2].axis('on')
plt.colorbar(axes[2].imshow(slice_yz, cmap='jet'), ax=axes[2])
plt.xlabel("Y-axis Length (cm)")
plt.ylabel("Z-axis Length (cm)")
           
plt.tight_layout()
plt.show()

In [ ]:
nonzero_mask = mean_energy_grid_record >= 1e-1
relative_std_err_energy_grid_record= np.zeros_like(std_err_energy_grid_record)
relative_std_err_energy_grid_record[nonzero_mask] = std_err_energy_grid_record[nonzero_mask] / mean_energy_grid_record[nonzero_mask] * 100

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

energy_grid = relative_std_err_energy_grid_record

mid0 = energy_grid.shape[0] // 2
mid1 = energy_grid.shape[1] // 2
mid2 = energy_grid.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid[mid0, :, :]  # plano YZ (fixando X)

# Plotar os planos com colormap
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# XY
axes[0].imshow(slice_xy, cmap='jet', origin='lower', aspect='auto')
axes[0].set_title("Plano XY (Z fixo)")
axes[0].axis('on')
plt.colorbar(axes[0].imshow(slice_xy, cmap='jet'), ax=axes[0])

# XZ
axes[1].imshow(slice_xz, cmap='jet', origin='lower', aspect='auto')
axes[1].set_title("Plano XZ (Y fixo)")
axes[1].axis('on')
plt.colorbar(axes[1].imshow(slice_xz, cmap='jet'), ax=axes[1])

# YZ
axes[2].imshow(slice_yz, cmap='jet', origin='lower', aspect='auto')
axes[2].set_title("Plano YZ (X fixo)")
axes[2].axis('on')
plt.colorbar(axes[2].imshow(slice_yz, cmap='jet'), ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# --- Parâmetros do voxel e volume ---
voxel_size = 0.1  # cm = 1 mm
volume_dims = (100, 100, 100)  # 10x10x10 cm³ => (100x100x100 voxels)
energy_grid = np.zeros(volume_dims)  # matriz 3D [x, y, z]

# --- Tabela simplificada de coeficientes para Pb ---
energy_levels = np.array([5E-02,1E-01,5E-01,1E00])
mu_total = np.array([7.787E00, 5.336E+00, 1.499E-01, 6.803E-02])     # cm²/g
mu_photo = np.array([7.292E+00, 5.237E+00, 8.257E-02, 1.810E-02])
mu_compton = np.array([9.478E-02 , 9.894E-02, 6.734E-02, 4.993E-02])
mu_pair = np.array([0, 0, 0, 0])

rho = 11.34  # g/cm³ Pb

# --- Tabela simplificada de coeficientes para H2O ---
energy_levels = np.array([5E-02,1E-01,5E-01,1E00])
mu_total = np.array([2.076E-01 , 1.654E-01, 9.665E-02 , 7.066E-02 ])     # cm²/g
mu_photo = np.array([2.725E-02, 2.763E-03, 1.883E-05, 3.681E-06])
mu_compton = np.array([1.803E-01 , 1.626E-01, 9.663E-02, 7.066E-02])
mu_pair = np.array([0, 0, 0, 0])

rho = 1  # g/cm³ Pb

# --- Interpoladores ---
get_mu_total = interp1d(energy_levels, mu_total, fill_value="extrapolate")
get_mu_photo = interp1d(energy_levels, mu_photo, fill_value="extrapolate")
get_mu_compton = interp1d(energy_levels, mu_compton, fill_value="extrapolate")
get_mu_pair = interp1d(energy_levels, mu_pair, fill_value="extrapolate")

def sample_compton_angle(E):
    mec2 = 0.511
    while True:
        cos_theta = np.random.uniform(-1, 1)
        theta = np.arccos(cos_theta)
        E_prime = E / (1 + (E / mec2) * (1 - cos_theta))
        P = E_prime**2 / E**2 * (E / E_prime + E_prime / E - np.sin(theta)**2)
        if np.random.uniform(0, 2) < P:
            return theta

def position_to_voxel(pos):
    idx = np.floor(pos / voxel_size).astype(int)
    if np.any(idx < 0) or np.any(idx >= np.array(volume_dims)):
        return None  # fora dos limites
    return tuple(idx)

# --- Inicialização do fóton ---
E0 = 1.0  # MeV
photon_energy = E0
photon_pos = np.array([5, 5, 0.0])  # cm
photon_dir = np.array([0.0, 0.0, 1.0])  # para z+

for N in range(10000):
    
    #print(N)
    photon_init = [(np.array([5.0, 5.0, 0.0]),np.array([0.0, 0.0, 1.0])),
                   (np.array([5.0, 5.0, 10.0]),np.array([0.0, 0.0, -1.0])),
                   (np.array([5.0, 10.0, 5.0]),np.array([0.0, -1.0, 0.0])),
                   (np.array([5.0, 0.0, 5.0]),np.array([0.0, 1.0, 0.0])),
                   (np.array([5.0, 10.0, 10.0]),np.array([0.0, -1.0, -1.0])),
                   (np.array([5.0, 0, 10.0]),np.array([0.0, 1.0, -1.0])),
                   (np.array([5.0, 10.0, 0.0]),np.array([0.0, -1.0, 1.0])),
                   (np.array([5.0, 0.0, 0.0]),np.array([0.0, 1.0, 1.0]))
                  ]
    for (photon_pos, photon_dir) in photon_init:

        E0 = 1.0  # MeV
        photon_energy = E0
        #photon_pos = np.array([5, 5, 0.0])  # cm
        #photon_dir = np.array([0.0, 0.0, 1.0])  # para z+
        
        while (0 <= photon_pos[0] <= volume_dims[0] * voxel_size) and (0 <= photon_pos[1] <= volume_dims[1] * voxel_size) and (0 <= photon_pos[2] <= volume_dims[2] * voxel_size) and photon_energy > 0.01:
            mu_tot = get_mu_total(photon_energy) * rho
            mu_ph = get_mu_photo(photon_energy)
            mu_cpt = get_mu_compton(photon_energy)
            mu_ppr = get_mu_pair(photon_energy)
            probs = np.array([mu_ph, mu_cpt, mu_ppr]) / (mu_ph + mu_cpt + mu_ppr)
        
            step = np.random.exponential(1 / mu_tot)
            photon_pos += photon_dir * step
        
            interaction = np.random.choice(['photo', 'compton', 'pair'], p=probs)
            voxel = position_to_voxel(photon_pos)
            if voxel is None:
                break
        
            if interaction == 'photo':
                energy_grid[voxel] += photon_energy
                break
        
            elif interaction == 'compton':
                theta = sample_compton_angle(photon_energy)
                phi = np.random.uniform(0, 2*np.pi)
                mec2 = 0.511
                E = photon_energy
                E_prime = E / (1 + (E / mec2) * (1 - np.cos(theta)))
                dE = E - E_prime
                energy_grid[voxel] += dE
                photon_energy = E_prime
        
                # Atualizar direção em 3D
                u = photon_dir
                if np.allclose(u, [0, 0, 1]) or np.allclose(u, [0, 0, -1]):
                    v = np.array([1, 0, 0])
                else:
                    v = np.cross(u, [0, 0, 1])
                    v /= np.linalg.norm(v)
                w = np.cross(u, v)
                new_dir = (np.sin(theta) * np.cos(phi) * v +
                           np.sin(theta) * np.sin(phi) * w +
                           np.cos(theta) * u)
                photon_dir = new_dir / np.linalg.norm(new_dir)
        
            elif interaction == 'pair':
                if photon_energy > 1.022:
                    deposited = photon_energy - 1.022
                    energy_grid[voxel] += deposited
                    break

# --- Visualização: Perfil de dose ao longo de profundidade (z) ---
depth_profile = energy_grid.sum(axis=(0, 1))  # soma em x e y
plt.figure(figsize=(6, 4))

z_axis = np.arange(volume_dims[2]) * voxel_size
plt.plot(z_axis, depth_profile)
plt.xlabel('Profundidade z [cm]')
plt.ylabel('Energia depositada [MeV]')
plt.title('Perfil de energia depositada (1 fóton)')
plt.grid(True)
plt.show()


In [ ]:
mid_axis1 = energy_grid.shape[0] // 2
mid_axis2 = energy_grid.shape[1] // 2
slice_1d = energy_grid[mid_axis1, mid_axis2, :]

y_axis = np.arange(volume_dims[1]) * voxel_size
plt.plot(y_axis, slice_1d/np.max(slice_1d))
plt.xlabel('Profundidade y [cm]')
plt.ylabel('Energia depositada [MeV]')
plt.title('Perfil de energia depositada (1 fóton)')
# plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mid0 = energy_grid.shape[0] // 2
mid1 = energy_grid.shape[1] // 2
mid2 = energy_grid.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid[mid0, :, :]  # plano YZ (fixando X)

# Plotar os planos com colormap
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# XY
axes[0].imshow(slice_xy, cmap='jet', origin='lower', aspect='auto')
axes[0].set_title("Plano XY (Z fixo)")
axes[0].axis('on')
plt.colorbar(axes[0].imshow(slice_xy, cmap='jet'), ax=axes[0])

# XZ
axes[1].imshow(slice_xz, cmap='jet', origin='lower', aspect='auto')
axes[1].set_title("Plano XZ (Y fixo)")
axes[1].axis('on')
plt.colorbar(axes[1].imshow(slice_xz, cmap='jet'), ax=axes[1])

# YZ
axes[2].imshow(slice_yz, cmap='jet', origin='lower', aspect='auto')
axes[2].set_title("Plano YZ (X fixo)")
axes[2].axis('on')
plt.colorbar(axes[2].imshow(slice_yz, cmap='jet'), ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

energy_grid_log = energy_grid + 1e-6
mid0 = energy_grid.shape[0] // 2
mid1 = energy_grid.shape[1] // 2
mid2 = energy_grid.shape[2] // 4

# Obter os 3 planos 2D
slice_xy = energy_grid[:, :, mid2]  # plano XY (fixando Z)
slice_xz = energy_grid[:, mid1, :]  # plano XZ (fixando Y)
slice_yz = energy_grid[mid0, :, :]  # plano YZ (fixando X)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Usar LogNorm para escala logarítmica
norm = LogNorm(vmin=np.min(energy_grid_log), vmax=np.max(energy_grid_log))

# XY
im0 = axes[0].imshow(slice_xy, cmap='jet', norm=norm, origin='lower', aspect='auto')
axes[0].set_title("Plano XY (Z fixo)")
plt.colorbar(im0, ax=axes[0])

# XZ
im1 = axes[1].imshow(slice_xz, cmap='jet', norm=norm, origin='lower', aspect='auto')
axes[1].set_title("Plano XZ (Y fixo)")
plt.colorbar(im1, ax=axes[1])

# YZ
im2 = axes[2].imshow(slice_yz, cmap='jet', norm=norm, origin='lower', aspect='auto')
axes[2].set_title("Plano YZ (X fixo)")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()